# Bài 1:

In [1]:
import Lab3.SVM as SVM 
import numpy as np
import cv2
import pandas as pd
import os
from Lab3.SVM import SVM

data_path = "data/chest_xray"

def load_data(split: str = "train", max_per_class: int | None = None) -> pd.DataFrame:
    split_dir = os.path.join(data_path, split)
    if not os.path.isdir(split_dir):
        raise FileNotFoundError(f"Không tìm thấy thư mục: {split_dir}")

    rows = []

    for label in os.listdir(split_dir):
        class_dir = os.path.join(split_dir, label)
        if not os.path.isdir(class_dir) or label.startswith("__"):
            continue

        class_count = 0

        for fname in os.listdir(class_dir):
            if max_per_class is not None and class_count >= max_per_class:
                break

            fpath = os.path.join(class_dir, fname)
            if not os.path.isfile(fpath):
                continue

            img = cv2.imread(fpath)
            if img is None:
                continue
            # Chuẩn hóa ảnh và trích xuất đặc trưng
            img = cv2.resize(img, (128, 128))
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            mean_bgr = img.mean(axis=(0, 1))
            std_gray = gray.std()
            hist = cv2.calcHist([gray], [0], None, [16], [0, 256]).flatten()
            hist = hist / (hist.sum() + 1e-8)

            row = {
                "file_name": fname,
                "label": label,
                "split": split,
                "mean_b": float(mean_bgr[0]),
                "mean_g": float(mean_bgr[1]),
                "mean_r": float(mean_bgr[2]),
                "std_gray": float(std_gray),
            }

            for i, v in enumerate(hist):
                row[f"hist_{i}"] = float(v)

            rows.append(row)
            class_count += 1

    return pd.DataFrame(rows)

In [2]:
train_df = load_data("train")
val_df = load_data("val")
test_df = load_data("test")

In [3]:
feature_cols = [c for c in train_df.columns if c not in ["file_name", "label", "split"]]

X_train = train_df[feature_cols].to_numpy(dtype=float)
X_val = val_df[feature_cols].to_numpy(dtype=float)
X_test = test_df[feature_cols].to_numpy(dtype=float)

y_train = train_df["label"].map({"NORMAL": -1, "PNEUMONIA": 1}).to_numpy(dtype=int)
y_val = val_df["label"].map({"NORMAL": -1, "PNEUMONIA": 1}).to_numpy(dtype=int)
y_test = test_df["label"].map({"NORMAL": -1, "PNEUMONIA": 1}).to_numpy(dtype=int)

print("Số features:", len(feature_cols))
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val  :", X_val.shape, "y_val  :", y_val.shape)
print("X_test :", X_test.shape, "y_test :", y_test.shape)

So features: 20
X_train: (5216, 20) y_train: (5216,)
X_val  : (16, 20) y_val  : (16,)
X_test : (624, 20) y_test : (624,)


In [4]:
model = SVM(lr=0.001, epochs=1000, C=1.0)

print("\nĐang huấn luyện mô hình SVM...")
model.fit(X_train, y_train, show_progress=True)
print("Hoàn thành huấn luyện!\n")

metrics = model.evaluate_metrics(X_test, y_test)

print(f"{'Accuracy':<20}{metrics['accuracy']:>20.4f}")
print(f"{'Precision':<20}{metrics['precision']:>20.4f}")
print(f"{'Recall':<20}{metrics['recall']:>20.4f}")
print(f"{'F1-score':<20}{metrics['f1_score']:>20.4f}")


Đang huấn luyện mô hình SVM...


Training SVM:   0%|          | 0/1000 [00:00<?, ?epoch/s]

Hoàn thành huấn luyện!

Accuracy                          0.6250
Precision                         0.6250
Recall                            1.0000
F1-score                          0.7692


# Bài 2:

In [6]:
import sklearn
import sklearn.svm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

sklearn_model = sklearn.svm.SVC(kernel="linear", C=1.0)
sklearn_model.fit(X_train, y_train)
y_pred = sklearn_model.predict(X_test)
sklearn_metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
}
print(f"{'Kết quả mô hình':<20}{'SVM Tự Code':>20}{'SVM Sklearn':>20}")
print(f"{'Accuracy':<20}{metrics['accuracy']:>20.4f}{sklearn_metrics['accuracy']:>20.4f}")
print(f"{'Precision':<20}{metrics['precision']:>20.4f}{sklearn_metrics['precision']:>20.4f}")
print(f"{'Recall':<20}{metrics['recall']:>20.4f}{sklearn_metrics['recall']:>20.4f}")
print(f"{'F1-score':<20}{metrics['f1_score']:>20.4f}{sklearn_metrics['f1_score']:>20.4f}")


Kết quả mô hình              SVM Tự Code         SVM Sklearn
Accuracy                          0.6250              0.6298
Precision                         0.6250              0.6280
Recall                            1.0000              1.0000
F1-score                          0.7692              0.7715
